## 2. Imports and Global Settings


In [12]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.float_format = "{:,.3f}".format

TRADING_DAYS = 252
START_DATE = "2018-01-01"
END_DATE = "2026-07-03"
RANDOM_SEED = 42

CURRENT_VOL_WINDOW = 63      # about 3m
EXPECTED_VOL_WINDOW = 42     # about 2m
SIGNAL_HISTORY_WINDOW = 252  # about 1y
PORTFOLIO_VOL_WINDOW = 756   # about 3y
FLOW_LOOKBACK_DAYS = 10      # about 2w
FORECAST_HORIZON_DAYS = 5    # about 1w
N_SIM_PATHS = 200

TARGET_PORTFOLIO_VOL = 0.10


## 3. Universe

FX sign convention:

- `USD` means a USD index proxy. Positive position means long USD basket.
- `USDJPY` positive means long USD versus JPY.
- `AUDUSD`, `EURUSD`, `GBPUSD` positive means long the non-USD base currency versus USD.

For FX, this prototype assumes fixed USD notional ADV because the available field is price only. In production, this can be replaced by internally assumed spot/liquidity tiers or external ADV estimates.


In [13]:
ASSET_CLASS_WEIGHTS = {
    "Rates": 0.45,
    "Equity": 0.35,
    "FX": 0.20,
}

UNIT_SCALERS = {
    # Output unit: USD mn DV01
    "Rates": 0.35,
    # Output unit: USD mn notional
    "Equity": 150.0,
    "FX": 120.0,
}

SIGNAL_CONFIGS = {
    "fast": {
        "ewma_pairs": ((4, 16), (8, 32), (16, 64), (32, 128)),
        "norm_window": 126,
        "response_scale": 1.0,
    },
    "medium": {
        "ewma_pairs": ((8, 32), (16, 64), (32, 128), (64, 256)),
        "norm_window": 252,
        "response_scale": 1.5,
    },
    "slow": {
        "ewma_pairs": ((16, 64), (32, 128), (64, 256), (128, 512)),
        "norm_window": 252,
        "response_scale": 2.0,
    },
}

ASSET_CLASS_SIGNAL_CONFIG = {
    "Rates": "medium",
    "Equity": "fast",
    "FX": "medium",
}

# Paste single-asset research results here when we have them.
ASSET_SIGNAL_CONFIG_OVERRIDES = {
    # "US10Y": {"ewma_pairs": ((4, 16), (8, 32), (16, 64), (32, 128)), "norm_window": 126, "response_scale": 1.0},
}

UNIVERSE = pd.DataFrame(
    [
        # Rates futures
        ("US2Y", "Rates", "US", "TU generic", "USD mn DV01", 1, np.nan, np.nan, np.nan),
        ("US5Y", "Rates", "US", "FV generic", "USD mn DV01", 4, np.nan, np.nan, np.nan),
        ("US10Y", "Rates", "US", "TY generic", "USD mn DV01", 8, np.nan, np.nan, np.nan),
        ("US20Y", "Rates", "US", "UXY generic", "USD mn DV01", 4, np.nan, np.nan, np.nan),
        ("US30Y", "Rates", "US", "US generic", "USD mn DV01", 4, np.nan, np.nan, np.nan),
        ("AU10Y", "Rates", "AU", "XM generic", "USD mn DV01", 2, np.nan, np.nan, np.nan),
        ("JP10Y", "Rates", "JP", "JB generic", "USD mn DV01", 4, np.nan, np.nan, np.nan),
        ("EU10Y", "Rates", "EU", "RX generic", "USD mn DV01", 8, np.nan, np.nan, np.nan),
        ("GB10Y", "Rates", "GB", "G generic", "USD mn DV01", 4, np.nan, np.nan, np.nan),

        # Equity index futures
        ("SPX", "Equity", "US", "ES generic", "USD mn notional", 8, 50.0, 1.00, np.nan),
        ("ASX200", "Equity", "AU", "XP generic", "USD mn notional", 3, 25.0, 0.67, np.nan),
        ("ESTX50", "Equity", "EU", "VG generic", "USD mn notional", 6, 10.0, 1.08, np.nan),
        ("FTSE100", "Equity", "GB", "Z generic", "USD mn notional", 4, 10.0, 1.27, np.nan),
        ("NKY225", "Equity", "JP", "NK generic", "USD mn notional", 5, 500.0, 0.0068, np.nan),

        # FX
        ("USD", "FX", "US", "DXY proxy", "USD mn notional", 7, np.nan, np.nan, 70_000.0),
        ("USDJPY", "FX", "JP", "USDJPY spot", "USD mn notional", 8, np.nan, np.nan, 90_000.0),
        ("AUDUSD", "FX", "AU", "AUDUSD spot", "USD mn notional", 5, np.nan, np.nan, 45_000.0),
        ("EURUSD", "FX", "EU", "EURUSD spot", "USD mn notional", 8, np.nan, np.nan, 120_000.0),
        ("GBPUSD", "FX", "GB", "GBPUSD spot", "USD mn notional", 6, np.nan, np.nan, 55_000.0),
    ],
    columns=[
        "asset", "asset_class", "country", "contract", "position_unit",
        "liquidity_factor", "contract_multiplier", "quote_to_usd", "assumed_adv_usd_mn"
    ],
).set_index("asset")

UNIVERSE["asset_class_weight"] = UNIVERSE["asset_class"].map(ASSET_CLASS_WEIGHTS)
UNIVERSE["unit_scaler"] = UNIVERSE["asset_class"].map(UNIT_SCALERS)
UNIVERSE["signal_config"] = UNIVERSE["asset_class"].map(ASSET_CLASS_SIGNAL_CONFIG)

UNIVERSE


,asset_class,country,contract,position_unit,liquidity_factor,contract_multiplier,quote_to_usd,assumed_adv_usd_mn,asset_class_weight,unit_scaler,signal_config
asset,,,,,,,,,,,
US2Y,Rates,US,TU generic,USD mn DV01,1,NaN,NaN,NaN,0.450,0.350,medium
US5Y,Rates,US,FV generic,USD mn DV01,4,NaN,NaN,NaN,0.450,0.350,medium
US10Y,Rates,US,TY generic,USD mn DV01,8,NaN,NaN,NaN,0.450,0.350,medium
US20Y,Rates,US,UXY generic,USD mn DV01,4,NaN,NaN,NaN,0.450,0.350,medium
US30Y,Rates,US,US generic,USD mn DV01,4,NaN,NaN,NaN,0.450,0.350,medium
AU10Y,Rates,AU,XM generic,USD mn DV01,2,NaN,NaN,NaN,0.450,0.350,medium
JP10Y,Rates,JP,JB generic,USD mn DV01,4,NaN,NaN,NaN,0.450,0.350,medium
EU10Y,Rates,EU,RX generic,USD mn DV01,8,NaN,NaN,NaN,0.450,0.350,medium
GB10Y,Rates,GB,G generic,USD mn DV01,4,NaN,NaN,NaN,0.450,0.350,medium


## 4. Simulated Market Database

The production database should use the same schema:

```text
market[field][asset]
```

Required fields by asset class:

- Rates: `price`, `volume`, `oi`, `dv01`
- Equity: `price`, `volume`, `oi`
- FX: `price`

Rates ADV is computed as:

```text
ADV in USD mn DV01 = rolling_mean(volume * DV01_per_contract / 1e6)
```

Equity ADV is computed as:

```text
ADV in USD mn notional = rolling_mean(price * volume * multiplier * quote_to_usd / 1e6)
```

FX ADV is an assumption in USD mn notional.


In [14]:
def _make_price_path(
    dates: pd.DatetimeIndex,
    start_price: float,
    daily_vol: float,
    common_factor: np.ndarray,
    beta: float,
    rng: np.random.Generator,
    drift: float = 0.0,
    trend_strength: float = 0.08,
) -> pd.Series:
    n = len(dates)
    slow_cycle = trend_strength * np.sin(np.linspace(0, 8 * np.pi, n))
    idio = rng.normal(0, daily_vol, n)
    rets = drift / TRADING_DAYS + beta * common_factor + idio + slow_cycle / TRADING_DAYS
    path = start_price * np.exp(np.cumsum(rets))
    return pd.Series(path, index=dates)


def _simulate_activity(
    dates: pd.DatetimeIndex,
    base: float,
    rng: np.random.Generator,
    vol: float = 0.25,
    seasonality: float = 0.15,
) -> pd.Series:
    n = len(dates)
    seasonal = 1.0 + seasonality * np.sin(np.linspace(0, 10 * np.pi, n))
    noise = rng.lognormal(mean=0.0, sigma=vol, size=n)
    return pd.Series(base * seasonal * noise, index=dates)


def simulate_market_database(
    universe: pd.DataFrame = UNIVERSE,
    start: str = START_DATE,
    end: str = END_DATE,
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start=start, end=end)
    n = len(dates)

    # Correlated macro factors. Signs are chosen to create plausible cross-asset regimes.
    global_risk = rng.normal(0, 0.006, n)
    rates_factor = rng.normal(0, 0.004, n) - 0.25 * global_risk
    usd_factor = rng.normal(0, 0.0035, n) - 0.15 * global_risk

    start_prices = {
        "US2Y": 102.0, "US5Y": 108.0, "US10Y": 112.0, "US20Y": 125.0, "US30Y": 132.0,
        "AU10Y": 96.0, "JP10Y": 148.0, "EU10Y": 137.0, "GB10Y": 101.0,
        "SPX": 2700.0, "ASX200": 6100.0, "ESTX50": 3400.0, "FTSE100": 7200.0, "NKY225": 22000.0,
        "USD": 94.0, "USDJPY": 110.0, "AUDUSD": 0.75, "EURUSD": 1.18, "GBPUSD": 1.35,
    }
    daily_vol = {
        "US2Y": 0.0018, "US5Y": 0.0028, "US10Y": 0.0038, "US20Y": 0.0046, "US30Y": 0.0052,
        "AU10Y": 0.0038, "JP10Y": 0.0022, "EU10Y": 0.0036, "GB10Y": 0.0042,
        "SPX": 0.010, "ASX200": 0.009, "ESTX50": 0.012, "FTSE100": 0.010, "NKY225": 0.011,
        "USD": 0.0045, "USDJPY": 0.0060, "AUDUSD": 0.0070, "EURUSD": 0.0055, "GBPUSD": 0.0065,
    }
    factor_map = {
        "Rates": rates_factor,
        "Equity": global_risk,
        "FX": usd_factor,
    }
    beta_map = {
        "Rates": 0.55,
        "Equity": 0.85,
        "FX": 0.65,
    }

    volume_base = {
        "US2Y": 480_000, "US5Y": 900_000, "US10Y": 1_450_000, "US20Y": 320_000, "US30Y": 230_000,
        "AU10Y": 250_000, "JP10Y": 45_000, "EU10Y": 780_000, "GB10Y": 250_000,
        "SPX": 1_700_000, "ASX200": 95_000, "ESTX50": 850_000, "FTSE100": 90_000, "NKY225": 80_000,
    }
    oi_base = {
        "US2Y": 4_000_000, "US5Y": 5_200_000, "US10Y": 4_600_000, "US20Y": 900_000, "US30Y": 1_100_000,
        "AU10Y": 900_000, "JP10Y": 160_000, "EU10Y": 2_400_000, "GB10Y": 850_000,
        "SPX": 2_900_000, "ASX200": 180_000, "ESTX50": 2_100_000, "FTSE100": 230_000, "NKY225": 310_000,
    }
    dv01_base = {
        # USD per bp per contract. These are intentionally calibrated to produce
        # ADV in USD mn DV01 near the broad magnitudes shown in the UBS tables.
        "US2Y": 40.0, "US5Y": 55.0, "US10Y": 110.0, "US20Y": 230.0, "US30Y": 340.0,
        "AU10Y": 78.0, "JP10Y": 1_280.0, "EU10Y": 185.0, "GB10Y": 145.0,
    }

    prices = pd.DataFrame(index=dates, columns=universe.index, dtype=float)
    volumes = pd.DataFrame(index=dates, columns=universe.index, dtype=float)
    oi = pd.DataFrame(index=dates, columns=universe.index, dtype=float)
    dv01 = pd.DataFrame(index=dates, columns=universe.index, dtype=float)

    for asset, row in universe.iterrows():
        asset_class = row["asset_class"]
        prices[asset] = _make_price_path(
            dates=dates,
            start_price=start_prices[asset],
            daily_vol=daily_vol[asset],
            common_factor=factor_map[asset_class],
            beta=beta_map[asset_class],
            rng=rng,
            drift=0.015 if asset_class == "Equity" else 0.0,
            trend_strength=0.10 if asset_class != "FX" else 0.06,
        )

        if asset_class in {"Rates", "Equity"}:
            volumes[asset] = _simulate_activity(dates, volume_base[asset], rng, vol=0.35)
            oi[asset] = _simulate_activity(dates, oi_base[asset], rng, vol=0.18)

        if asset_class == "Rates":
            dv01_noise = rng.normal(0, 0.025, len(dates))
            # DV01 moves slowly and mildly with the futures price level.
            dv01[asset] = dv01_base[asset] * (prices[asset] / prices[asset].iloc[0]) ** 0.15 * (1 + dv01_noise)

    market = pd.concat(
        {
            "price": prices,
            "volume": volumes,
            "oi": oi,
            "dv01": dv01,
        },
        axis=1,
    )
    market.columns.names = ["field", "asset"]
    return market


market = simulate_market_database()
market.tail()


field        price                                                          \
asset         US2Y    US5Y   US10Y   US20Y   US30Y   AU10Y   JP10Y   EU10Y   
2026-06-29 117.489 177.398 184.796 124.243 147.497 145.283 166.694 128.980   
2026-06-30 117.433 177.493 185.216 124.174 146.713 144.446 167.049 128.430   
2026-07-01 117.367 177.747 184.422 124.270 147.408 144.699 167.329 128.435   
2026-07-02 116.971 176.750 184.482 124.855 146.874 144.047 167.245 128.767   
2026-07-03 116.699 176.054 184.601 124.185 146.267 143.804 167.227 128.386   

field                        ... dv01                                          \
asset       GB10Y       SPX  ...  SPX ASX200 ESTX50 FTSE100 NKY225 USD USDJPY   
2026-06-29 76.566 1,439.319  ...  NaN    NaN    NaN     NaN    NaN NaN    NaN   
2026-06-30 76.124 1,430.699  ...  NaN    NaN    NaN     NaN    NaN NaN    NaN   
2026-07-01 76.027 1,424.953  ...  NaN    NaN    NaN     NaN    NaN NaN    NaN   
2026-07-02 75.264 1,421.570  ...  NaN    NaN    NaN     NaN    NaN NaN    NaN   
2026-07-03 75.373 1,406.568  ...  NaN    NaN    NaN     NaN    NaN NaN    NaN   

field                            
asset      AUDUSD EURUSD GBPUSD  
2026-06-29    NaN    NaN    NaN  
2026-06-30    NaN    NaN    NaN  
2026-07-01    NaN    NaN    NaN  
2026-07-02    NaN    NaN    NaN  
2026-07-03    NaN    NaN    NaN  

[5 rows x 76 columns]

## 5. Core CTA Engine


In [15]:
def get_field(market: pd.DataFrame, field: str) -> pd.DataFrame:
    return market[field].copy()


def get_signal_config(asset: str, universe: pd.DataFrame = UNIVERSE) -> dict:
    if asset in ASSET_SIGNAL_CONFIG_OVERRIDES:
        return ASSET_SIGNAL_CONFIG_OVERRIDES[asset]
    config_name = universe.loc[asset, "signal_config"]
    return SIGNAL_CONFIGS[config_name]


def annualized_realized_vol(price_df: pd.DataFrame, window: int, floor: float = 0.02) -> pd.DataFrame:
    vol = price_df.pct_change().rolling(window, min_periods=max(20, window // 2)).std() * np.sqrt(TRADING_DAYS)
    return vol.clip(lower=floor)


def momentum_signal(
    price_df: pd.DataFrame,
    ewma_pairs: Iterable[Tuple[int, int]],
    norm_window: int,
    response_scale: float,
) -> pd.DataFrame:
    # UBS-style EWMA crossover signal with a transparent tanh response approximation.
    ret_vol = price_df.pct_change().rolling(norm_window, min_periods=max(40, norm_window // 2)).std().clip(lower=1e-6)
    trend_parts = []

    for short_span, long_span in ewma_pairs:
        fast = price_df.ewm(span=short_span, adjust=False, min_periods=short_span).mean()
        slow = price_df.ewm(span=long_span, adjust=False, min_periods=long_span).mean()
        spread = fast - slow

        # Convert price spread into a dimensionless trend-in-sigma style quantity.
        denom = price_df * ret_vol * np.sqrt(long_span)
        trend_parts.append(spread / denom)

    trend_stack = pd.concat(trend_parts, axis=1, keys=range(len(trend_parts)))
    trend = trend_stack.T.groupby(level=1).mean().T
    signal = np.tanh(trend / response_scale).clip(-1, 1)
    signal.columns = price_df.columns
    return signal


def build_signal_matrix(price_df: pd.DataFrame, universe: pd.DataFrame = UNIVERSE) -> pd.DataFrame:
    parts = []
    for asset in price_df.columns:
        parts.append(momentum_signal(price_df[[asset]], **get_signal_config(asset, universe)))
    return pd.concat(parts, axis=1).reindex(columns=price_df.columns)


def compute_adv(
    price_df: pd.DataFrame,
    volume_df: pd.DataFrame,
    dv01_df: pd.DataFrame,
    universe: pd.DataFrame = UNIVERSE,
    window: int = 63,
) -> pd.DataFrame:
    adv = pd.DataFrame(index=price_df.index, columns=price_df.columns, dtype=float)

    for asset, row in universe.iterrows():
        asset_class = row["asset_class"]

        if asset_class == "Rates":
            # Same unit as rates positions: USD mn DV01.
            adv[asset] = (volume_df[asset] * dv01_df[asset] / 1_000_000).rolling(window, min_periods=20).mean()

        elif asset_class == "Equity":
            # Same unit as equity positions: USD mn notional.
            adv[asset] = (
                price_df[asset]
                * volume_df[asset]
                * row["contract_multiplier"]
                * row["quote_to_usd"]
                / 1_000_000
            ).rolling(window, min_periods=20).mean()

        elif asset_class == "FX":
            # Same unit as FX positions: USD mn notional. No volume field required.
            adv[asset] = float(row["assumed_adv_usd_mn"])

    return adv


def compute_portfolio_scaling(
    signal: pd.DataFrame,
    vol: pd.DataFrame,
    returns: pd.DataFrame,
    universe: pd.DataFrame = UNIVERSE,
    target_vol: float = TARGET_PORTFOLIO_VOL,
    window: int = PORTFOLIO_VOL_WINDOW,
) -> pd.Series:
    liq = universe["liquidity_factor"].astype(float).reindex(signal.columns)
    acw = universe["asset_class_weight"].astype(float).reindex(signal.columns)
    pseudo_weight = signal.multiply(liq * acw, axis=1).divide(vol).replace([np.inf, -np.inf], np.nan).fillna(0)
    pseudo_weight = pseudo_weight.div(pseudo_weight.abs().sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    proxy_port_ret = (pseudo_weight.shift(1) * returns.fillna(0)).sum(axis=1)

    realised = proxy_port_ret.rolling(window, min_periods=252).std() * np.sqrt(TRADING_DAYS)
    scaling = (target_vol / realised).replace([np.inf, -np.inf], np.nan)
    return scaling.clip(lower=0.25, upper=4.0).ffill().fillna(1.0)


def compute_positions(
    signal: pd.DataFrame,
    vol: pd.DataFrame,
    universe: pd.DataFrame = UNIVERSE,
    portfolio_scaling: pd.Series | float = 1.0,
) -> pd.DataFrame:
    liq = universe["liquidity_factor"].astype(float).reindex(signal.columns)
    acw = universe["asset_class_weight"].astype(float).reindex(signal.columns)
    unit_scaler = universe["unit_scaler"].astype(float).reindex(signal.columns)

    raw = signal.multiply(liq * acw * unit_scaler, axis=1).divide(vol)

    if isinstance(portfolio_scaling, pd.Series):
        out = raw.multiply(portfolio_scaling, axis=0)
    else:
        out = raw * float(portfolio_scaling)

    return out.replace([np.inf, -np.inf], np.nan)


def latest_non_null(df: pd.DataFrame) -> pd.Series:
    return df.dropna(how="all").iloc[-1]


## 6. Per-Asset Signal Parameter Search

This is our added research/calibration layer.

UBS describes the signal construction methodology, but the paper does not disclose an exact production parameter set for every market. For our tracker, we can therefore run a transparent per-asset grid search to decide whether each market behaves better with faster or slower trend horizons.

Important distinction:

- This grid search is **not** the CTA positioning model itself.
- It is a way to choose the `ewma_pairs`, `norm_window`, and `response_scale` that feed the UBS-style positioning model.
- In production, this should be done with train/test or walk-forward validation to avoid overfitting.

The default score below combines:

```text
annualized excess return versus buy-and-hold
+ Sharpe improvement
+ drawdown improvement
+ directional hit-rate edge
```

For the final product, we should treat this as a diagnostic table and prefer robust parameter regions over one-off winners.


In [16]:
PAIRSET_GRID = {
    "fast": ((4, 16), (8, 32), (16, 64), (32, 128)),
    "medium": ((8, 32), (16, 64), (32, 128), (64, 256)),
    "slow": ((16, 64), (32, 128), (64, 256), (128, 512)),
}

PARAMETER_GRID = {
    "pairset": ["fast", "medium", "slow"],
    "norm_window": [126, 252],
    "response_scale": [0.75, 1.00, 1.50, 2.00],
}

TRAIN_FRACTION = 0.70
MIN_ACTIVE_EXPOSURE = 0.10


def _ann_return(ret: pd.Series) -> float:
    ret = ret.dropna()
    if ret.empty:
        return np.nan
    return (1 + ret).prod() ** (TRADING_DAYS / len(ret)) - 1


def _sharpe(ret: pd.Series) -> float:
    ret = ret.dropna()
    if ret.std() == 0 or ret.empty:
        return np.nan
    return ret.mean() / ret.std() * np.sqrt(TRADING_DAYS)


def _max_drawdown(ret: pd.Series) -> float:
    ret = ret.dropna()
    if ret.empty:
        return np.nan
    wealth = (1 + ret).cumprod()
    return (wealth / wealth.cummax() - 1).min()


def _directional_hit_rate(exposure: pd.Series, ret: pd.Series, min_active: float = MIN_ACTIVE_EXPOSURE) -> float:
    aligned = pd.concat([exposure, ret], axis=1).dropna()
    if aligned.empty:
        return np.nan
    x = aligned.iloc[:, 0]
    r = aligned.iloc[:, 1]
    active = x.abs() >= min_active
    if active.sum() == 0:
        return np.nan
    return (np.sign(x[active]) * r[active] > 0).mean()


def evaluate_signal_config(
    price: pd.Series,
    config: dict,
    train_fraction: float = TRAIN_FRACTION,
) -> dict:
    price_df = price.to_frame(price.name)
    sig = momentum_signal(price_df, **config).iloc[:, 0]

    asset_ret = price.pct_change()
    exposure = sig.shift(1).clip(-1, 1)
    strategy_ret = exposure * asset_ret
    buy_hold_ret = asset_ret

    valid = pd.concat([strategy_ret, buy_hold_ret, exposure], axis=1).dropna()
    valid.columns = ["strategy", "buy_hold", "exposure"]
    if len(valid) < 300:
        return {}

    split = int(len(valid) * train_fraction)
    train = valid.iloc[:split]
    test = valid.iloc[split:]

    def score_block(block: pd.DataFrame, prefix: str) -> dict:
        ann_strategy = _ann_return(block["strategy"])
        ann_bh = _ann_return(block["buy_hold"])
        sharpe_strategy = _sharpe(block["strategy"])
        sharpe_bh = _sharpe(block["buy_hold"])
        dd_strategy = _max_drawdown(block["strategy"])
        dd_bh = _max_drawdown(block["buy_hold"])
        hit = _directional_hit_rate(block["exposure"], block["buy_hold"])

        ann_excess = ann_strategy - ann_bh
        sharpe_excess = sharpe_strategy - sharpe_bh
        dd_improvement = abs(dd_bh) - abs(dd_strategy)
        hit_edge = hit - 0.5

        composite = (
            100.0 * ann_excess
            + sharpe_excess
            + 5.0 * dd_improvement
            + 2.0 * hit_edge
        )

        return {
            f"{prefix}_ann_strategy": ann_strategy,
            f"{prefix}_ann_buy_hold": ann_bh,
            f"{prefix}_ann_excess": ann_excess,
            f"{prefix}_sharpe_strategy": sharpe_strategy,
            f"{prefix}_sharpe_buy_hold": sharpe_bh,
            f"{prefix}_sharpe_excess": sharpe_excess,
            f"{prefix}_max_dd_strategy": dd_strategy,
            f"{prefix}_max_dd_buy_hold": dd_bh,
            f"{prefix}_dd_improvement": dd_improvement,
            f"{prefix}_hit_rate": hit,
            f"{prefix}_score": composite,
        }

    out = {}
    out.update(score_block(train, "train"))
    out.update(score_block(test, "test"))
    out.update(score_block(valid, "full"))
    out["train_start"] = train.index[0]
    out["train_end"] = train.index[-1]
    out["test_start"] = test.index[0]
    out["test_end"] = test.index[-1]
    return out


def run_asset_parameter_grid(price_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for asset in price_df.columns:
        price = price_df[asset].dropna()
        for pairset in PARAMETER_GRID["pairset"]:
            for norm_window in PARAMETER_GRID["norm_window"]:
                for response_scale in PARAMETER_GRID["response_scale"]:
                    config = {
                        "ewma_pairs": PAIRSET_GRID[pairset],
                        "norm_window": norm_window,
                        "response_scale": response_scale,
                    }
                    metrics = evaluate_signal_config(price, config)
                    if not metrics:
                        continue
                    rows.append({
                        "asset": asset,
                        "asset_class": UNIVERSE.loc[asset, "asset_class"],
                        "pairset": pairset,
                        "ewma_pairs": PAIRSET_GRID[pairset],
                        "norm_window": norm_window,
                        "response_scale": response_scale,
                        **metrics,
                    })

    results = pd.DataFrame(rows)
    return results.sort_values(["asset", "train_score"], ascending=[True, False])


prices = get_field(market, "price")
asset_grid_results = run_asset_parameter_grid(prices)

best_signal_table = (
    asset_grid_results
    .sort_values(["asset", "train_score"], ascending=[True, False])
    .groupby("asset", as_index=False)
    .head(1)
    .set_index("asset")
    .sort_values(["asset_class", "train_score"], ascending=[True, False])
)

ASSET_SIGNAL_CONFIG_OVERRIDES = {
    asset: {
        "ewma_pairs": row["ewma_pairs"],
        "norm_window": int(row["norm_window"]),
        "response_scale": float(row["response_scale"]),
    }
    for asset, row in best_signal_table.iterrows()
}

display_cols = [
    "asset_class", "pairset", "norm_window", "response_scale",
    "train_score", "test_score", "full_score",
    "train_ann_excess", "test_ann_excess",
    "train_sharpe_excess", "test_sharpe_excess",
    "train_dd_improvement", "test_dd_improvement",
    "train_hit_rate", "test_hit_rate",
]

best_signal_table[display_cols]


,asset_class,pairset,norm_window,response_scale,train_score,test_score,full_score,train_ann_excess,test_ann_excess,train_sharpe_excess,test_sharpe_excess,train_dd_improvement,test_dd_improvement,train_hit_rate,test_hit_rate
asset,,,,,,,,,,,,,,,
SPX,Equity,fast,126,2.000,11.127,9.232,11.541,0.087,0.075,0.413,0.388,0.396,0.274,0.515,0.500
ASX200,Equity,fast,252,2.000,9.255,8.229,9.667,0.069,0.076,0.349,-0.493,0.408,0.254,0.491,0.447
FTSE100,Equity,fast,252,2.000,7.180,-5.876,4.006,0.051,-0.062,0.053,-0.599,0.400,0.170,0.511,0.513
NKY225,Equity,fast,252,0.750,6.114,8.288,6.840,0.038,0.061,0.338,0.506,0.383,0.335,0.524,0.502
ESTX50,Equity,fast,126,2.000,1.695,33.732,13.440,0.008,0.288,-0.435,2.025,0.277,0.564,0.482,0.520
EURUSD,FX,fast,126,0.750,8.915,-5.897,5.051,0.067,-0.058,0.855,-0.625,0.272,0.112,0.509,0.502
USDJPY,FX,medium,126,2.000,3.640,3.649,3.887,0.019,0.026,0.152,0.188,0.311,0.149,0.499,0.538
GBPUSD,FX,fast,126,0.750,-0.314,-4.306,-1.494,-0.015,-0.031,0.167,-1.206,0.192,0.016,0.511,0.454
USD,FX,slow,126,2.000,-4.225,-0.040,-2.874,-0.041,-0.004,-0.628,-0.315,0.113,0.149,0.485,0.482


## 7. ADV and Cross-Asset Crowding Logic

ADV means **average daily volume**.

In this tracker, ADV is not just “number of contracts traded”. It must be converted into the same economic unit as the position before we compute `%ADV`.

Why this matters:

- A `USD mn DV01` rates position cannot be compared directly with a `USD mn notional` equity or FX position.
- But `Position / ADV` is comparable because it asks: how large is this CTA position relative to the amount the market usually trades in one day?

UBS uses this logic to identify markets more vulnerable to CTA flows. A small absolute position can be crowded if the market is illiquid; a large absolute position can be less risky if the market has huge daily volume.

The formulas used here are:

```text
Rates ADV in USD mn DV01
= rolling_mean(volume_contracts * DV01_per_contract / 1e6)

Equity ADV in USD mn notional
= rolling_mean(price * volume_contracts * contract_multiplier * FX_conversion / 1e6)

FX ADV in USD mn notional
= assumed spot-market ADV
```

Then:

```text
Position %ADV = Position / ADV_same_unit * 100
Flow %ADV     = Flow / ADV_same_unit * 100
```

This is why UBS can say things like one bond future has lower absolute DV01 positioning but higher `%ADV`: it trades less every day, so CTA flows can matter more.


## 8. Current Positioning and Recent Flow


In [17]:
prices = get_field(market, "price")
volumes = get_field(market, "volume")
oi = get_field(market, "oi")
dv01 = get_field(market, "dv01")
returns = prices.pct_change()

signal = build_signal_matrix(prices)
current_vol = annualized_realized_vol(prices, CURRENT_VOL_WINDOW)
expected_vol = annualized_realized_vol(prices, EXPECTED_VOL_WINDOW)
adv = compute_adv(prices, volumes, dv01)
portfolio_scaling = compute_portfolio_scaling(signal, current_vol, returns)
positions = compute_positions(signal, current_vol, portfolio_scaling=portfolio_scaling)

current_date = positions.dropna(how="all").index[-1]
comparison_date = positions.index[positions.index.get_loc(current_date) - FLOW_LOOKBACK_DAYS]

current_position = positions.loc[current_date]
past_position = positions.loc[comparison_date]
recent_flow = current_position - past_position
current_adv = adv.loc[current_date]
current_signal = signal.loc[current_date]
current_vol_latest = current_vol.loc[current_date]

current_table = pd.DataFrame({
    "asset_class": UNIVERSE["asset_class"],
    "country": UNIVERSE["country"],
    "contract": UNIVERSE["contract"],
    "signal": current_signal,
    "vol_3m": current_vol_latest,
    "position": current_position,
    "position_unit": UNIVERSE["position_unit"],
    "adv_same_unit": current_adv,
    "position_%ADV": current_position / current_adv * 100,
    "flow_2w": recent_flow,
    "flow_2w_%ADV": recent_flow / current_adv * 100,
    "liquidity_factor": UNIVERSE["liquidity_factor"],
    "portfolio_scaling": portfolio_scaling.loc[current_date],
})

current_table = current_table.sort_values("position_%ADV", key=lambda s: s.abs(), ascending=False)
print(f"Current date: {current_date.date()} | 2w comparison date: {comparison_date.date()}")
current_table


Current date: 2026-07-03 | 2w comparison date: 2026-06-19


,asset_class,country,contract,signal,vol_3m,position,position_unit,adv_same_unit,position_%ADV,flow_2w,flow_2w_%ADV,liquidity_factor,portfolio_scaling
asset,,,,,,,,,,,,,
NKY225,Equity,JP,NK generic,0.481,0.164,"2,295.002",USD mn notional,"5,256.246",43.662,24.263,0.462,5,2.983
JP10Y,Rates,JP,JB generic,-0.706,0.057,-23.392,USD mn DV01,62.975,-37.145,2.676,4.249,4,2.983
GB10Y,Rates,GB,G generic,0.383,0.079,9.092,USD mn DV01,33.745,26.944,-3.588,-10.632,4,2.983
US2Y,Rates,US,TU generic,-0.420,0.051,-3.889,USD mn DV01,21.008,-18.512,-1.028,-4.894,1,2.983
EU10Y,Rates,EU,RX generic,-0.403,0.068,-22.359,USD mn DV01,144.696,-15.452,-0.908,-0.628,8,2.983
US10Y,Rates,US,TY generic,-0.325,0.067,-18.306,USD mn DV01,167.400,-10.935,2.674,1.598,8,2.983
US30Y,Rates,US,US generic,-0.307,0.101,-5.740,USD mn DV01,77.635,-7.394,-3.783,-4.873,4,2.983
US20Y,Rates,US,UXY generic,-0.188,0.085,-4.145,USD mn DV01,72.189,-5.742,-0.516,-0.715,4,2.983
ESTX50,Equity,EU,VG generic,0.162,0.222,686.363,USD mn notional,"14,993.488",4.578,"1,258.440",8.393,6,2.983


## 9. One-Week Expected Flow Forecast

UBS forecasts future CTA positioning by forecasting the components of the same positioning formula.

For the next week:

- expected signal: Monte Carlo price paths, then recompute the nonlinear path-dependent signal,
- expected volatility: current 2-month realized volatility,
- liquidity factor: held constant,
- portfolio volatility scaling: held constant.

This is a CTA flow forecast, not a return forecast.


In [18]:
def forecast_signal_monte_carlo(
    price_df: pd.DataFrame,
    asset: str,
    horizon_days: int = FORECAST_HORIZON_DAYS,
    n_paths: int = N_SIM_PATHS,
    history_window_bd: int = 504,
    seed: int = RANDOM_SEED,
) -> float:
    rng = np.random.default_rng(seed + abs(hash(asset)) % 10_000)
    full_hist = price_df[[asset]].dropna()
    hist = full_hist.tail(history_window_bd)
    recent_returns = hist[asset].pct_change().dropna().tail(EXPECTED_VOL_WINDOW)

    sigma_daily = max(float(recent_returns.std()), 1e-6)
    last_price = float(hist[asset].iloc[-1])
    config = get_signal_config(asset)

    terminal_signals = []
    for _ in range(n_paths):
        shocks = rng.normal(loc=0.0, scale=sigma_daily, size=horizon_days)
        sim_prices = last_price * np.exp(np.cumsum(shocks))
        future_index = pd.bdate_range(hist.index[-1] + pd.offsets.BDay(1), periods=horizon_days)
        sim = pd.concat([hist, pd.DataFrame({asset: sim_prices}, index=future_index)])
        terminal_signals.append(momentum_signal(sim, **config).iloc[-1, 0])

    return float(np.nanmean(terminal_signals))


expected_signal = pd.Series(
    {
        asset: forecast_signal_monte_carlo(prices, asset)
        for asset in prices.columns
    },
    name=current_date + pd.offsets.BDay(FORECAST_HORIZON_DAYS),
)

expected_position = compute_positions(
    pd.DataFrame([expected_signal], index=[expected_signal.name]),
    pd.DataFrame([expected_vol.loc[current_date]], index=[expected_signal.name]),
    portfolio_scaling=float(portfolio_scaling.loc[current_date]),
).iloc[0]

expected_flow = expected_position - current_position

forecast_table = current_table.copy()
forecast_table["expected_signal_1w"] = expected_signal
forecast_table["expected_position_1w"] = expected_position
forecast_table["expected_flow_1w"] = expected_flow
forecast_table["expected_flow_1w_%ADV"] = expected_flow / current_adv * 100
forecast_table["abs_expected_flow_1w_%ADV"] = forecast_table["expected_flow_1w_%ADV"].abs()
forecast_table = forecast_table.sort_values("abs_expected_flow_1w_%ADV", ascending=False)

forecast_table


,asset_class,country,contract,signal,vol_3m,position,position_unit,adv_same_unit,position_%ADV,flow_2w,flow_2w_%ADV,liquidity_factor,portfolio_scaling,expected_signal_1w,expected_position_1w,expected_flow_1w,expected_flow_1w_%ADV,abs_expected_flow_1w_%ADV
asset,,,,,,,,,,,,,,,,,,
GB10Y,Rates,GB,G generic,0.383,0.079,9.092,USD mn DV01,33.745,26.944,-3.588,-10.632,4,2.983,0.293,7.220,-1.872,-5.548,5.548
US10Y,Rates,US,TY generic,-0.325,0.067,-18.306,USD mn DV01,167.400,-10.935,2.674,1.598,8,2.983,-0.459,-25.346,-7.041,-4.206,4.206
US5Y,Rates,US,FV generic,0.070,0.055,2.382,USD mn DV01,57.865,4.117,-4.736,-8.185,4,2.983,0.002,0.078,-2.304,-3.981,3.981
AU10Y,Rates,AU,XM generic,0.011,0.075,0.137,USD mn DV01,20.345,0.675,-0.356,-1.749,2,2.983,-0.020,-0.275,-0.413,-2.028,2.028
JP10Y,Rates,JP,JB generic,-0.706,0.057,-23.392,USD mn DV01,62.975,-37.145,2.676,4.249,4,2.983,-0.616,-22.136,1.256,1.994,1.994
ESTX50,Equity,EU,VG generic,0.162,0.222,686.363,USD mn notional,"14,993.488",4.578,"1,258.440",8.393,6,2.983,0.123,480.257,-206.106,-1.375,1.375
US2Y,Rates,US,TU generic,-0.420,0.051,-3.889,USD mn DV01,21.008,-18.512,-1.028,-4.894,1,2.983,-0.426,-4.155,-0.266,-1.266,1.266
ASX200,Equity,AU,XP generic,-0.009,0.170,-23.543,USD mn notional,"5,283.789",-0.446,17.996,0.341,3,2.983,-0.023,-71.984,-48.440,-0.917,0.917
US20Y,Rates,US,UXY generic,-0.188,0.085,-4.145,USD mn DV01,72.189,-5.742,-0.516,-0.715,4,2.983,-0.175,-3.714,0.431,0.597,0.597


## 10. Dashboard Views


In [19]:
summary_by_class = (
    forecast_table
    .assign(
        abs_position_pct_adv=lambda x: x["position_%ADV"].abs(),
        abs_flow_pct_adv=lambda x: x["expected_flow_1w_%ADV"].abs(),
    )
    .groupby("asset_class")
    .agg(
        assets=("country", "count"),
        avg_abs_position_pct_adv=("abs_position_pct_adv", "mean"),
        max_abs_position_pct_adv=("abs_position_pct_adv", "max"),
        avg_abs_expected_flow_pct_adv=("abs_flow_pct_adv", "mean"),
        max_abs_expected_flow_pct_adv=("abs_flow_pct_adv", "max"),
    )
    .sort_values("max_abs_expected_flow_pct_adv", ascending=False)
)

summary_by_class


,assets,avg_abs_position_pct_adv,max_abs_position_pct_adv,avg_abs_expected_flow_pct_adv,max_abs_expected_flow_pct_adv
asset_class,,,,,
Rates,9,14.102,37.145,2.199,5.548
Equity,5,10.507,43.662,0.653,1.375
FX,5,1.473,4.183,0.157,0.369


In [20]:
plot_df = forecast_table.reset_index(names="asset").sort_values("position_%ADV")

fig = px.bar(
    plot_df,
    x="position_%ADV",
    y="asset",
    color="asset_class",
    orientation="h",
    title="Current CTA Positioning by Asset (%ADV, same-unit ADV)",
    hover_data=["country", "contract", "signal", "position", "position_unit", "adv_same_unit"],
)
fig.add_vline(x=0, line_width=1, line_color="black")
fig.update_layout(height=620, xaxis_title="Current position (%ADV)", yaxis_title="")
fig.show()


In [21]:
flow_df = forecast_table.reset_index(names="asset").sort_values("expected_flow_1w_%ADV")

fig = px.bar(
    flow_df,
    x="expected_flow_1w_%ADV",
    y="asset",
    color="asset_class",
    orientation="h",
    title="Expected CTA Flow Over Next Week (%ADV)",
    hover_data=["country", "contract", "expected_signal_1w", "expected_flow_1w", "position_unit"],
)
fig.add_vline(x=0, line_width=1, line_color="black")
fig.update_layout(height=620, xaxis_title="Expected 1w flow (%ADV)", yaxis_title="")
fig.show()


In [22]:
def plot_asset_tracker(asset: str = "US10Y", lookback_days: int = 504):
    hist_idx = prices.index[-lookback_days:]
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=(
            f"{asset} price",
            "Momentum signal",
            f"CTA position ({UNIVERSE.loc[asset, 'position_unit']})",
        ),
    )
    fig.add_trace(go.Scatter(x=hist_idx, y=prices.loc[hist_idx, asset], name="price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=hist_idx, y=signal.loc[hist_idx, asset], name="signal"), row=2, col=1)
    fig.add_hline(y=0, row=2, col=1, line_width=1, line_color="black")
    fig.add_trace(go.Scatter(x=hist_idx, y=positions.loc[hist_idx, asset], name="position"), row=3, col=1)
    fig.add_hline(y=0, row=3, col=1, line_width=1, line_color="black")
    fig.update_layout(height=760, title=f"{asset} CTA Tracker")
    fig.show()


plot_asset_tracker("US10Y")


## 11. Sanity Checks and Export Hooks


In [23]:
def run_sanity_checks(
    market: pd.DataFrame,
    universe: pd.DataFrame,
    current_table: pd.DataFrame,
    forecast_table: pd.DataFrame,
) -> None:
    fields = set(market.columns.get_level_values("field"))
    required_fields = {"price", "volume", "oi", "dv01"}
    missing_fields = required_fields - fields
    if missing_fields:
        raise ValueError(f"Missing top-level fields: {missing_fields}")

    price = get_field(market, "price")
    if price[universe.index].isna().all().any():
        missing = price.columns[price.isna().all()].tolist()
        raise ValueError(f"Assets with no price data: {missing}")

    rates = universe.index[universe["asset_class"].eq("Rates")]
    for field in ["volume", "oi", "dv01"]:
        df = get_field(market, field)
        missing = [asset for asset in rates if df[asset].isna().all()]
        if missing:
            raise ValueError(f"Rates missing {field}: {missing}")

    equity = universe.index[universe["asset_class"].eq("Equity")]
    for field in ["volume", "oi"]:
        df = get_field(market, field)
        missing = [asset for asset in equity if df[asset].isna().all()]
        if missing:
            raise ValueError(f"Equity missing {field}: {missing}")

    if current_table["position_%ADV"].replace([np.inf, -np.inf], np.nan).isna().any():
        bad = current_table.index[current_table["position_%ADV"].isna()].tolist()
        raise ValueError(f"Current %ADV missing for: {bad}")

    if forecast_table["expected_flow_1w_%ADV"].replace([np.inf, -np.inf], np.nan).isna().any():
        bad = forecast_table.index[forecast_table["expected_flow_1w_%ADV"].isna()].tolist()
        raise ValueError(f"Forecast %ADV missing for: {bad}")

    print("Sanity checks passed.")


run_sanity_checks(market, UNIVERSE, current_table, forecast_table)

# Export examples. Uncomment when needed.
# current_table.to_csv("cta_current_positioning.csv")
# forecast_table.to_csv("cta_expected_flow_1w.csv")


Sanity checks passed.


## 12. Production Replacement Checklist

When the real database is available, replace only `simulate_market_database()` with a real loader that returns the same `market[field][asset]` schema.

Checklist:

1. **Rates**: confirm `DV01` is per contract and in USD per bp. Then `volume * DV01 / 1e6` gives USD mn DV01 ADV.
2. **Equity**: confirm each contract multiplier and quote-to-USD conversion.
3. **FX**: decide internal ADV assumptions by pair, or source external ADV estimates.
4. **Roll adjustment**: for real generic futures, use roll-adjusted price series for signals and returns.
5. **Signal calibration**: run single-asset train/test or walk-forward research, then paste asset-specific settings into `ASSET_SIGNAL_CONFIG_OVERRIDES`.
6. **Response calibration**: upgrade from transparent `tanh` to an empirical response function if we want to match UBS's “near uniform signal distribution” more closely.
7. **Validation**: compare current positions and recent flows with CFTC/IMM, broker positioning, and known CTA flow episodes.
